# 📊 交互式数据分析与可视化 (Interactive EDA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuler/explore-colab/blob/main/notebooks/02_interactive_data_visualizer.ipynb)

> 本 Notebook 演示在 Google Colab 中使用交互式控件（ipywidgets）、数据表格过滤（Colab Data Table）以及动态可视化图表。

## 1. 启用 Colab 交互式数据表扩展
Colab 内置了对 Pandas DataFrame 的交互式浏览增强插件，支持在单元格内直接搜索、分页与排序。

In [ ]:
# 开启 Colab 原生数据表插件（在 Colab 环境下生效）
try:
    %load_ext google.colab.data_table
    print('✅ Colab data_table 扩展加载成功')
except Exception as e:
    print('ℹ️ 本地环境或不支持 google.colab 模块，使用标准 Pandas 显示')


## 2. 生成模拟时序与多维特征数据

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n_samples = 1000
dates = pd.date_range('2026-01-01', periods=n_samples, freq='h')

df = pd.DataFrame({
    'timestamp': dates,
    'device_id': np.random.choice(['edge-node-A', 'edge-node-B', 'edge-node-C', 'cloud-gpu-01'], size=n_samples),
    'temperature_c': np.random.normal(loc=65.0, scale=8.5, size=n_samples).round(2),
    'gpu_utilization_pct': np.clip(np.random.normal(loc=72.0, scale=18.0, size=n_samples), 0, 100).round(1),
    'memory_used_gb': np.clip(np.random.normal(loc=12.0, scale=3.2, size=n_samples), 1.0, 24.0).round(2),
    'throughput_req_per_s': np.random.poisson(lam=450, size=n_samples)
})

print(f'数据样本总计: {len(df)} 行')
df.head(10)


## 3. 交互式控件与动态绘图 (ipywidgets + Matplotlib)
拖动滑动条即可动态改变计算参数与图形展示。

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

def plot_device_trend(selected_device, min_temp):
    filtered = df[(df['device_id'] == selected_device) & (df['temperature_c'] >= min_temp)]
    fig, ax1 = plt.subplots(figsize=(12, 4))
    
    ax1.plot(filtered['timestamp'], filtered['gpu_utilization_pct'], label='GPU 利用率 (%)', color='#2563eb', alpha=0.8)
    ax1.set_ylabel('GPU 利用率 (%)', color='#2563eb')
    ax1.set_ylim(0, 105)
    
    ax2 = ax1.twinx()
    ax2.plot(filtered['timestamp'], filtered['temperature_c'], label='温度 (°C)', color='#dc2626', linestyle='--', alpha=0.7)
    ax2.set_ylabel('温度 (°C)', color='#dc2626')
    
    plt.title(f'设备实时监控: {selected_device} (筛选: 温度 >= {min_temp}°C, 共 {len(filtered)} 条记录)')
    plt.show()

devices = list(df['device_id'].unique())
interact(plot_device_trend, 
         selected_device=Dropdown(options=devices, value=devices[0], description='选择设备:'),
         min_temp=FloatSlider(min=40, max=90, step=2, value=50, description='最低温度:'))


## 4. 统计相关性矩阵热力图

In [ ]:
numeric_cols = ['temperature_c', 'gpu_utilization_pct', 'memory_used_gb', 'throughput_req_per_s']
corr = df[numeric_cols].corr()

plt.figure(figsize=(7, 5))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Pearson 相关系数')
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=30, ha='right')
plt.yticks(range(len(numeric_cols)), numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        plt.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', color='black')
plt.title('指标相关性矩阵分析')
plt.tight_layout()
plt.show()
